# 03 — Deep Feature Extraction with WideResNet-50

**Author:** Sherif Abd El-Rady

**Objective:** Extract multi-scale patch embeddings from a frozen WideResNet-50-2 backbone, visualise the feature space with t-SNE, and quantify the separation between normal and defect embeddings.

## Why pretrained CNN features for anomaly detection?

Training a dedicated anomaly detector from scratch requires many defect examples — which we don't have (that's the whole problem). Instead, PatchCore exploits the observation that **ImageNet-pretrained features are transferable**: they encode texture, shape, and structural patterns that generalise well to industrial surfaces, even without any fine-tuning.

## Which layers?

| Layer | Stride | Receptive field | Feature type |
|---|---|---|---|
| `layer1` | 4 | ~11×11 | Fine textures |
| **`layer2`** | **8** | **~27×27** | **Local patterns** |
| **`layer3`** | **16** | **~67×67** | **Structural context** |
| `layer4` | 32 | Large | Semantic (too abstract) |

PatchCore uses `layer2 + layer3` — concatenated after bilinear alignment to `layer2` resolution.


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import FeatureExtractor

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC    = ROOT / 'data' / 'synthetic'
    CATEGORY = ''
    print('Using synthetic dataset')

base      = MVTEC / CATEGORY if CATEGORY else MVTEC
train_dir = base / 'train' / 'good'
test_dir  = base / 'test'

print(f'Device: {DEVICE}')

In [ ]:
# ── 3.1  Initialise extractor and validate hook shapes ────────────────────────
extractor = FeatureExtractor(
    layers=['layer2', 'layer3'],
    input_size=224,
    device=DEVICE,
)

# Feed one image to check shapes
dummy = (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
batch = extractor.preprocess([dummy])
patches, spatial = extractor.extract_patches(batch)

print(f'Input:              {tuple(batch.shape)}')
print(f'Patch embeddings:   {patches.shape}  (n_patches × C_total)')
print(f'Feature map (H,W):  {spatial}')
print(f'C_total = {patches.shape[1]}  (layer2: 512, layer3: 1024 → both aligned + concatenated)')

In [ ]:
# ── 3.2  Extract features from training images (batch processing) ─────────────
train_paths = sorted(train_dir.glob('*.png')) + sorted(train_dir.glob('*.jpg'))
BATCH_SIZE  = 8

all_train_patches = []
train_spatial = None

print(f'Extracting features from {len(train_paths)} training images...')
for i in tqdm(range(0, len(train_paths), BATCH_SIZE), desc='Training batches'):
    batch_paths = train_paths[i:i+BATCH_SIZE]
    imgs  = [cv2.imread(str(p)) for p in batch_paths]
    imgs  = [img for img in imgs if img is not None]
    if not imgs: continue
    batch = extractor.preprocess(imgs)
    patches, spatial = extractor.extract_patches(batch)
    all_train_patches.append(patches)
    train_spatial = spatial

train_patches = np.vstack(all_train_patches)
print(f'\nTotal training patches: {train_patches.shape[0]:,}')
print(f'Feature dimension:      {train_patches.shape[1]}')
print(f'Spatial resolution:     {train_spatial}')

In [ ]:
# ── 3.3  Extract features from a sample of test images ────────────────────────
test_imgs_by_label = {'normal': [], 'defect': []}
test_patch_list = []
test_labels     = []
MAX_PER_CLASS   = 5  # limit for t-SNE speed

for cls_dir in sorted(test_dir.iterdir()):
    if not cls_dir.is_dir(): continue
    label = 'normal' if cls_dir.name == 'good' else 'defect'
    paths = (sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg')))[:MAX_PER_CLASS]
    for p in paths:
        img = cv2.imread(str(p))
        if img is None: continue
        batch = extractor.preprocess([img])
        patches, _ = extractor.extract_patches(batch)
        test_patch_list.append(patches)
        test_labels.extend([0 if label == 'normal' else 1] * len(patches))

test_patches = np.vstack(test_patch_list)
test_labels  = np.array(test_labels)
print(f'Test patches: {test_patches.shape[0]:,}  ({(test_labels==0).sum()} normal / {(test_labels==1).sum()} defect)')

In [ ]:
# ── 3.4  t-SNE visualisation of the feature space ────────────────────────────
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import numpy as np

# Sample for t-SNE (it's O(n²))
N_TSNE = min(2000, len(train_patches), len(test_patches))
rng    = np.random.default_rng(42)

train_idx = rng.choice(len(train_patches), N_TSNE // 2, replace=False)
test_idx  = rng.choice(len(test_patches),  N_TSNE // 2, replace=False)

combined  = np.vstack([train_patches[train_idx], test_patches[test_idx]])
split_at  = len(train_idx)
test_cols = np.array(['#e74c3c' if test_labels[i] == 1 else '#3498db' for i in test_idx])

# PCA first for speed (t-SNE on 1536-D is slow)
pca      = PCA(n_components=50, random_state=42)
combined_pca = pca.fit_transform(combined)
var_explained = pca.explained_variance_ratio_.sum()
print(f'PCA 50 components explain {var_explained:.1%} of variance')

tsne     = TSNE(n_components=2, perplexity=40, n_iter=500, random_state=42)
embedded = tsne.fit_transform(combined_pca)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(embedded[:split_at, 0], embedded[:split_at, 1],
           c='#2ecc71', alpha=0.4, s=8, label='Normal (train)')
ax.scatter(embedded[split_at:, 0][test_labels[test_idx] == 0],
           embedded[split_at:, 1][test_labels[test_idx] == 0],
           c='#3498db', alpha=0.6, s=12, label='Normal (test)')
ax.scatter(embedded[split_at:, 0][test_labels[test_idx] == 1],
           embedded[split_at:, 1][test_labels[test_idx] == 1],
           c='#e74c3c', alpha=0.8, s=20, marker='^', label='Defect (test)')

ax.set_title('t-SNE projection of WideResNet-50-2 patch features\n'
             'layer2 + layer3 concatenated (1536-D → PCA 50 → t-SNE 2)',
             fontsize=10)
ax.legend(fontsize=9)
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.show()

print('\nObservation: if defect patches (red) lie away from the normal cluster,'
      ' k-NN distance will flag them correctly.')

In [ ]:
# ── 3.5  Feature map visualisation — what do the features look like spatially? ─
# Visualise the L2-norm of the feature map at each spatial position
# (a proxy for 'feature activation strength')

sample_path = sorted(train_dir.glob('*.png'))[0]
img_bgr = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

batch = extractor.preprocess([img_bgr])
with torch.no_grad():
    feat = extractor.extract(batch)   # (1, C, H_f, W_f)

feat_np = feat[0].cpu().numpy()  # (C, H_f, W_f)
activation = np.linalg.norm(feat_np, axis=0)  # (H_f, W_f)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_rgb)
axes[0].set_title('Input image')
axes[0].axis('off')

im = axes[1].imshow(activation, cmap='viridis')
plt.colorbar(im, ax=axes[1])
axes[1].set_title(f'Feature L2-norm\n({feat_np.shape[1]}×{feat_np.shape[2]} feature map)')
axes[1].axis('off')

act_resized = cv2.resize(activation, (img_rgb.shape[1], img_rgb.shape[0]))
act_norm = (act_resized - act_resized.min()) / (act_resized.max() - act_resized.min())
act_u8   = (act_norm * 255).astype(np.uint8)
heat     = cv2.applyColorMap(act_u8, cv2.COLORMAP_VIRIDIS)
overlay  = cv2.addWeighted(img_bgr, 0.5, heat, 0.5, 0)
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title('Feature activation overlay')
axes[2].axis('off')

plt.suptitle('WideResNet-50-2 Feature Activation Visualisation', fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Step | Result |
|---|---|
| Backbone | WideResNet-50-2 (ImageNet), layers 2+3 |
| Patch dim | 1536-D (512 + 1024 concatenated, aligned to layer2 resolution) |
| t-SNE | Defect patches visible outside normal cluster |
| Feature maps | Spatially meaningful — respond to surface structure |

## Next step

Open `04_memory_bank.ipynb` — build the coreset-sampled memory bank from training features.
